# LeetCode 1686 — Stone Game VI bằng Python

Link đề: https://leetcode.com/problems/stone-game-vi/ · Độ khó: **Medium** · Chủ đề: Greedy, Game Theory, Sorting, Heap (Priority Queue), Math

Skeleton LeetCode:

```python
class Solution:
    def stoneGameVI(self, aliceValues: List[int], bobValues: List[int]) -> int:
        ...
```

Lộ trình của notebook: **minimax vét cạn trên bitmask** (đáp án chuẩn cho `n` nhỏ) → nhận ra "lấy một viên = mình được điểm **và** đối thủ mất điểm" → **tham lam sắp xếp theo `a[i] + b[i]`** (`O(n log n)`) → **đếm theo tổng** vì giá trị ≤ 100 (`O(n + V)`) → phiên bản **heap** để thấy cùng một thứ tự ưu tiên.

## 1. Mục tiêu học

1. Mô hình hóa một trò chơi hai người **chơi tối ưu** bằng **minimax**: trạng thái = tập viên đá còn lại (bitmask), giá trị = hiệu điểm "người đi − người kia".
2. Hiểu vì sao khi **hai người định giá khác nhau**, viên đá "tốt nhất" để lấy không phải viên mình thích nhất, mà là viên có **`a[i] + b[i]` lớn nhất** (được cho mình + tước của đối thủ).
3. Chứng minh tham lam bằng **lập luận đổi chỗ (exchange argument)** và kiểm chứng bằng vét cạn trên input ngẫu nhiên.
4. Tối ưu sắp xếp thành **đếm theo khóa nhỏ** (counting sort) khi miền giá trị bị chặn.
5. Viết code nộp gọn với `sorted(..., reverse=True)`, slicing `[::2]`, `[1::2]` và trả về `1 / 0 / -1`.

## 2. Đề bài & ví dụ

Alice và Bob lần lượt chơi, **Alice đi trước**. Có `n` viên đá. Mỗi lượt, người chơi **bỏ đi một viên bất kỳ** còn lại và nhận điểm bằng giá trị của viên đó **theo cách định giá của chính mình**:
- `aliceValues[i]` là giá trị viên `i` đối với Alice,
- `bobValues[i]` là giá trị viên `i` đối với Bob.

Khi hết đá, ai nhiều điểm hơn thì thắng; bằng nhau là hòa. Cả hai **chơi tối ưu** và biết cách định giá của nhau. Trả về:
- `1` nếu Alice thắng, `-1` nếu Bob thắng, `0` nếu hòa.

**Ví dụ 1:** `aliceValues = [1,3], bobValues = [2,1]` → `1`. Alice lấy viên 1 (được 3), Bob chỉ còn viên 0 (được 2). Alice thắng.

**Ví dụ 2:** `aliceValues = [1,2], bobValues = [3,1]` → `0`. Alice lấy viên 0 (được 1), Bob lấy viên 1 (được 1). Hòa.

**Ví dụ 3:** `aliceValues = [2,4,3], bobValues = [1,6,7]` → `-1`. Dù Alice đi thế nào, Bob vẫn nhiều điểm hơn.

**Ràng buộc:**
- `n == aliceValues.length == bobValues.length`
- `1 ≤ n ≤ 10⁵`
- `1 ≤ aliceValues[i], bobValues[i] ≤ 100`

In [1]:
import random
import time
import heapq
from functools import lru_cache
from typing import List

examples = [
    (([1, 3], [2, 1]), 1),
    (([1, 2], [3, 1]), 0),
    (([2, 4, 3], [1, 6, 7]), -1),
]
for (a, b), want in examples:
    print(f"aliceValues={a}, bobValues={b} -> kỳ vọng {want}")

aliceValues=[1, 3], bobValues=[2, 1] -> kỳ vọng 1
aliceValues=[1, 2], bobValues=[3, 1] -> kỳ vọng 0
aliceValues=[2, 4, 3], bobValues=[1, 6, 7] -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1 Bitmask biểu diễn tập con

Với `n` viên đá, một số nguyên `mask` có `n` bit: bit `i` bằng 1 nghĩa là viên `i` **còn trên bàn**. Thao tác cơ bản:
- kiểm tra viên `i` còn không: `mask >> i & 1`
- bỏ viên `i`: `mask ^ (1 << i)` (hoặc `mask & ~(1 << i)`)
- tập đầy đủ: `(1 << n) - 1`

In [2]:
n = 4
full = (1 << n) - 1
print("full =", bin(full))
mask = full ^ (1 << 2)          # bỏ viên 2
print("sau khi bỏ viên 2:", bin(mask), [i for i in range(n) if mask >> i & 1])
print("số viên còn lại:", bin(mask).count("1"))

full = 0b1111
sau khi bỏ viên 2: 0b1011 [0, 1, 3]
số viên còn lại: 3


### 3.2 `lru_cache` để ghi nhớ (memoization)

Hàm đệ quy theo `mask` sẽ bị gọi lặp lại rất nhiều lần với cùng `mask` (vì có nhiều thứ tự lấy dẫn tới cùng một tập còn lại). `@lru_cache(maxsize=None)` lưu kết quả theo tham số, mỗi `mask` chỉ tính một lần ⇒ tối đa `2ⁿ` trạng thái.

In [3]:
calls = 0
@lru_cache(maxsize=None)
def count_subsets_paths(mask):
    global calls
    calls += 1
    if mask == 0:
        return 1
    return sum(count_subsets_paths(mask ^ (1 << i)) for i in range(4) if mask >> i & 1)

print("số thứ tự lấy hết 4 viên =", count_subsets_paths(15), "(= 4! = 24)")
print("số lần hàm thật sự chạy =", calls, "(= 2^4 = 16 trạng thái)")

số thứ tự lấy hết 4 viên = 24 (= 4! = 24)
số lần hàm thật sự chạy = 16 (= 2^4 = 16 trạng thái)


### 3.3 Sắp xếp theo khóa và slicing bước nhảy

`sorted(idx, key=lambda i: a[i] + b[i], reverse=True)` cho danh sách chỉ số theo tổng giảm dần. Slicing `order[::2]` lấy các phần tử ở vị trí chẵn (0, 2, 4, …) — chính là các lượt của Alice; `order[1::2]` lấy vị trí lẻ — lượt của Bob.

In [4]:
a, b = [2, 4, 3], [1, 6, 7]
order = sorted(range(len(a)), key=lambda i: a[i] + b[i], reverse=True)
print("tổng a+b:", [a[i] + b[i] for i in range(len(a))])
print("thứ tự chọn:", order)
print("lượt Alice:", order[::2], " lượt Bob:", order[1::2])

tổng a+b: [3, 10, 10]
thứ tự chọn: [1, 2, 0]
lượt Alice: [1, 0]  lượt Bob: [2]


### 3.4 Hàm dấu (sign) và so sánh ba chiều

Đề trả về `1 / 0 / -1` theo dấu của hiệu `A − B`. Một mẹo Python: `(x > 0) - (x < 0)` cho đúng dấu của `x` vì `True == 1`, `False == 0`.

In [5]:
def sign(x):
    return (x > 0) - (x < 0)
print([sign(v) for v in [7, 0, -3]])

[1, 0, -1]


### 3.5 Heap (`heapq`) — hàng đợi ưu tiên

`heapq` là **min-heap**. Muốn lấy phần tử lớn nhất trước thì đẩy vào `-key`. `heapify` tạo heap trong `O(n)`, mỗi `heappop` tốn `O(log n)`.

In [6]:
h = [(-(x + y), i) for i, (x, y) in enumerate(zip([2, 4, 3], [1, 6, 7]))]
heapq.heapify(h)
while h:
    neg, i = heapq.heappop(h)
    print(f"lấy viên {i} có tổng {-neg}")

lấy viên 1 có tổng 10
lấy viên 2 có tổng 10
lấy viên 0 có tổng 3


## 4. Ý tưởng nền tảng / trực giác

Hãy tính tay **ví dụ 3**: `a = [2,4,3]`, `b = [1,6,7]`. Alice có 3 lựa chọn ở lượt đầu, Bob 2, Alice 1. Liệt kê toàn bộ cây trò chơi:

| Alice lấy trước | Bob chọn tốt nhất | Alice lấy cuối | Điểm A | Điểm B | Hiệu |
|---|---|---|---|---|---|
| viên 0 (A+2) | viên 2 (B+7) | viên 1 (A+4) | 6 | 7 | −1 |
| viên 1 (A+4) | viên 2 (B+7) | viên 0 (A+2) | 6 | 7 | −1 |
| viên 2 (A+3) | viên 1 (B+6) | viên 0 (A+2) | 5 | 6 | −1 |

Mọi nhánh đều cho Bob thắng ⇒ đáp án `-1`.

**Quan sát then chốt:** nếu Alice lấy viên 1 (viên cô thích nhất, giá 4) thì Bob lấy viên 2 (giá 7 với Bob). Nếu Alice lấy viên 2 (giá chỉ 3 với cô) thì cô **tước** của Bob 7 điểm. Mỗi viên khi Alice lấy làm thay đổi hiệu `A − B` thêm `a[i]` (mình được) **và** ngăn Bob lấy được `b[i]`. Nên "giá trị thật" của viên `i` với người lấy là `a[i] + b[i]`.

In [7]:
from itertools import permutations

def play_order(a, b, order):
    """Cho trước thứ tự lấy (Alice lượt chẵn, Bob lượt lẻ) -> (điểm A, điểm B)."""
    A = sum(a[i] for i in order[::2])
    B = sum(b[i] for i in order[1::2])
    return A, B

a, b = [2, 4, 3], [1, 6, 7]
for order in permutations(range(3)):
    A, B = play_order(a, b, list(order))
    print(order, "-> A =", A, ", B =", B, ", hiệu =", A - B)

(0, 1, 2) -> A = 5 , B = 6 , hiệu = -1
(0, 2, 1) -> A = 6 , B = 7 , hiệu = -1
(1, 0, 2) -> A = 7 , B = 1 , hiệu = 6
(1, 2, 0) -> A = 6 , B = 7 , hiệu = -1
(2, 0, 1) -> A = 7 , B = 1 , hiệu = 6
(2, 1, 0) -> A = 5 , B = 6 , hiệu = -1


Bảng trên liệt kê **mọi thứ tự**, nhưng không phải thứ tự nào cũng là "chơi tối ưu". Chơi tối ưu nghĩa là ở mỗi lượt, người đi chọn nước tốt nhất **giả sử đối thủ cũng chơi tốt nhất về sau** — đó là minimax ở mục 5.

## 5. Cách 1 — Brute Force / baseline: minimax trên bitmask

**Ý tưởng:** gọi `f(mask)` = hiệu điểm lớn nhất mà **người sắp đi** đạt được (điểm mình − điểm đối thủ) tính từ trạng thái `mask` trở về sau. Ai đi được suy ra từ số viên đã lấy: nếu số viên đã lấy chẵn thì là Alice.

- Nếu `mask == 0`: `f = 0`.
- Người đi chọn viên `i` còn lại, được `v_i` (theo cách định giá của mình), rồi đối thủ đạt `f(mask ^ bit_i)` theo góc nhìn **của đối thủ**. Nên:

`f(mask) = max over i ( v_i − f(mask without i) )`

Đáp án là `sign(f(full))` (từ góc nhìn của Alice).

> Vì sao tối đa hóa **hiệu** cũng là tối ưu theo nghĩa thắng/hòa/thua? Hàm `sign` là đơn điệu không giảm, mà `max`/`min` giao hoán với hàm đơn điệu, nên giá trị trò chơi theo thang thắng/hòa/thua đúng bằng `sign` của giá trị trò chơi theo hiệu.

In [8]:
def stone_game_brute(a, b):
    n = len(a)
    full = (1 << n) - 1

    @lru_cache(maxsize=None)
    def f(mask):
        if mask == 0:
            return 0
        taken = n - bin(mask).count("1")
        vals = a if taken % 2 == 0 else b       # lượt Alice nếu đã lấy chẵn viên
        best = -10**9
        for i in range(n):
            if mask >> i & 1:
                best = max(best, vals[i] - f(mask ^ (1 << i)))
        return best

    d = f(full)
    return (d > 0) - (d < 0)

for (a, b), want in examples:
    print(stone_game_brute(a, b), "kỳ vọng", want)

1 kỳ vọng 1
0 kỳ vọng 0
-1 kỳ vọng -1


**Độ phức tạp:** có `2ⁿ` trạng thái, mỗi trạng thái duyệt `n` viên ⇒ Time `O(n · 2ⁿ)`, Space `O(2ⁿ)`.

**Nút thắt:** `n = 20` đã là ~20 triệu phép; `n = 10⁵` thì không thể. Cần một quy tắc chọn **không phụ thuộc** vào việc tìm kiếm toàn bộ cây.

In [9]:
random.seed(1)
for n in [8, 12, 14]:
    a = [random.randint(1, 100) for _ in range(n)]
    b = [random.randint(1, 100) for _ in range(n)]
    t0 = time.perf_counter()
    stone_game_brute(a, b)
    print(f"n={n}: {time.perf_counter() - t0:.3f}s")

n=8: 0.000s
n=12: 0.005s
n=14: 0.021s


## 6. Ý tưởng thuật toán / bước đột phá

**Viết lại hiệu điểm.** Gọi `S_A` là tập viên Alice lấy, `S_B` là tập viên Bob lấy. Ta có

`A − B = Σ_{i∈S_A} a[i] − Σ_{j∈S_B} b[j]`.

Cộng và trừ `Σ_{tất cả} b[i]`:

`A − B = Σ_{i∈S_A} (a[i] + b[i]) − Σ_{mọi i} b[i]`.

Số hạng thứ hai là **hằng số**! Vậy Alice muốn **tối đa** `Σ_{i∈S_A} (a[i] + b[i])`. Tương tự (đối xứng):

`B − A = Σ_{j∈S_B} (a[j] + b[j]) − Σ_{mọi i} a[i]`

⇒ Bob muốn **tối đa** tổng `a[j] + b[j]` trên tập của mình. **Cả hai cùng tranh nhau các viên có `c[i] = a[i] + b[i]` lớn.**

**Quy tắc:** *Nếu đến lượt mình thì lấy viên có `a[i] + b[i]` lớn nhất còn lại.* Khi đó viên lớn nhất về Alice, viên lớn thứ hai về Bob, thứ ba về Alice, … — đúng thứ tự sắp xếp giảm dần theo `c`.

**Invariant:** sau mỗi lượt, tập viên còn lại luôn là "đuôi" của danh sách đã sắp xếp; mỗi người đang giữ đúng các viên ở vị trí có chẵn/lẻ tương ứng.

In [10]:
a, b = [2, 4, 3], [1, 6, 7]
c = [x + y for x, y in zip(a, b)]
print("c = a+b =", c)
SA = [2, 0]  # Alice lấy viên 2 (c=10) rồi viên 0 (c=3)
SB = [1]
lhs = sum(a[i] for i in SA) - sum(b[j] for j in SB)
rhs = sum(c[i] for i in SA) - sum(b)
print("A-B tính trực tiếp =", lhs, "; theo công thức =", rhs)

c = a+b = [3, 10, 10]
A-B tính trực tiếp = -1 ; theo công thức = -1


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam + sắp xếp theo `a[i] + b[i]`

Từng đoạn:
1. Tạo danh sách chỉ số `0..n-1`, sắp xếp giảm dần theo `a[i] + b[i]`.
2. Alice nhận các vị trí chẵn trong thứ tự: cộng `a[i]`.
3. Bob nhận các vị trí lẻ: cộng `b[i]`.
4. So sánh và trả về dấu.

Lưu ý: khi hai viên có **cùng tổng**, thứ tự giữa chúng **không ảnh hưởng kết quả** (công thức ở mục 6 chỉ phụ thuộc tổng `c` trên tập của Alice).

In [11]:
def stone_game_sort(a, b):
    order = sorted(range(len(a)), key=lambda i: a[i] + b[i], reverse=True)
    A = sum(a[i] for i in order[::2])
    B = sum(b[i] for i in order[1::2])
    return (A > B) - (A < B)

for (a, b), want in examples:
    print(stone_game_sort(a, b), "kỳ vọng", want)

1 kỳ vọng 1
0 kỳ vọng 0
-1 kỳ vọng -1


Code nộp LeetCode của Cách 2:

```python
class Solution:
    def stoneGameVI(self, aliceValues: List[int], bobValues: List[int]) -> int:
        order = sorted(range(len(aliceValues)),
                       key=lambda i: aliceValues[i] + bobValues[i], reverse=True)
        A = sum(aliceValues[i] for i in order[::2])
        B = sum(bobValues[i] for i in order[1::2])
        return (A > B) - (A < B)
```

**Độ phức tạp:** Time `O(n log n)` (sắp xếp), Space `O(n)` (mảng thứ tự).

### 7.2 Cách 3 — Đếm theo tổng (counting sort), `O(n + V)`

Vì `1 ≤ a[i], b[i] ≤ 100` nên `c[i] ∈ [2, 200]`. Thay vì sắp xếp, ta gom viên đá vào **201 cái xô** theo `c`. Duyệt xô từ 200 về 2; trong mỗi xô, các viên được lấy luân phiên (Alice, Bob, Alice, …) tiếp nối lượt hiện tại.

Từng đoạn:
1. `bucket[s]` = danh sách chỉ số có tổng `s`.
2. Biến `turn` (0 = Alice, 1 = Bob) chạy liên tục qua mọi xô.
3. Gặp viên `i`: nếu `turn == 0` cộng `a[i]` vào `A`, ngược lại cộng `b[i]` vào `B`; đảo `turn`.

In [12]:
def stone_game_count(a, b):
    bucket = [[] for _ in range(201)]
    for i, (x, y) in enumerate(zip(a, b)):
        bucket[x + y].append(i)
    A = B = 0
    turn = 0
    for s in range(200, 1, -1):
        for i in bucket[s]:
            if turn == 0:
                A += a[i]
            else:
                B += b[i]
            turn ^= 1
    return (A > B) - (A < B)

for (a, b), want in examples:
    print(stone_game_count(a, b), "kỳ vọng", want)

1 kỳ vọng 1
0 kỳ vọng 0
-1 kỳ vọng -1


Một biến thể gọn hơn: ta chỉ cần biết **trong mỗi xô Alice lấy bao nhiêu viên**. Nhưng vì các viên trong cùng xô có `a[i]` khác nhau, cách dễ nhất vẫn là duyệt từng viên (thứ tự trong xô tùy ý — xem mục 10 về vì sao không ảnh hưởng).

Code nộp LeetCode của Cách 3:

```python
class Solution:
    def stoneGameVI(self, aliceValues: List[int], bobValues: List[int]) -> int:
        bucket = [[] for _ in range(201)]
        for i, (x, y) in enumerate(zip(aliceValues, bobValues)):
            bucket[x + y].append(i)
        A = B = 0
        turn = 0
        for s in range(200, 1, -1):
            for i in bucket[s]:
                if turn == 0:
                    A += aliceValues[i]
                else:
                    B += bobValues[i]
                turn ^= 1
        return (A > B) - (A < B)
```

**Độ phức tạp:** Time `O(n + V)` với `V = 200`, Space `O(n + V)`.

### 7.3 Cách 4 — Mô phỏng từng lượt bằng heap

Cách này "kể chuyện" đúng như trò chơi: mỗi lượt, người đi rút viên có `a[i] + b[i]` lớn nhất khỏi heap. Không nhanh hơn Cách 2, nhưng hữu ích khi biến thể yêu cầu **thêm viên đá giữa chừng** hoặc **chỉ chơi k lượt**.

In [13]:
def stone_game_heap(a, b):
    h = [(-(x + y), i) for i, (x, y) in enumerate(zip(a, b))]
    heapq.heapify(h)
    A = B = 0
    alice = True
    while h:
        _, i = heapq.heappop(h)
        if alice:
            A += a[i]
        else:
            B += b[i]
        alice = not alice
    return (A > B) - (A < B)

for (a, b), want in examples:
    print(stone_game_heap(a, b), "kỳ vọng", want)

1 kỳ vọng 1
0 kỳ vọng 0
-1 kỳ vọng -1


**Độ phức tạp:** Time `O(n log n)` (heapify `O(n)` + `n` lần pop `O(log n)`), Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 3 `a = [2,4,3]`, `b = [1,6,7]`:

| bước | viên | c = a+b | lượt | cộng | A | B |
|---|---|---|---|---|---|---|
| 0 | 1 | 10 | Alice | a[1] = 4 | 4 | 0 |
| 1 | 2 | 10 | Bob | b[2] = 7 | 4 | 7 |
| 2 | 0 | 3 | Alice | a[0] = 2 | 6 | 7 |

`A = 6 < B = 7` ⇒ `-1`. (Viên 1 và 2 cùng tổng 10; `sorted` ổn định nên viên 1 đứng trước. Nếu đổi thứ tự hai viên này thì `A = 3 + 2 = 5`, `B = 6` — hiệu vẫn `−1`.)

In [14]:
def trace_sort(a, b):
    order = sorted(range(len(a)), key=lambda i: a[i] + b[i], reverse=True)
    print("thứ tự theo c giảm dần:", order, "c =", [a[i] + b[i] for i in order])
    A = B = 0
    for step, i in enumerate(order):
        who = "Alice" if step % 2 == 0 else "Bob"
        if step % 2 == 0:
            A += a[i]; gain = f"a[{i}]={a[i]}"
        else:
            B += b[i]; gain = f"b[{i}]={b[i]}"
        print(f"bước {step}: viên {i}, c={a[i] + b[i]:3d}, {who:5s} cộng {gain:8s} -> A={A}, B={B}")
    res = (A > B) - (A < B)
    print("kết quả:", res)
    return res

trace_sort([2, 4, 3], [1, 6, 7])

thứ tự theo c giảm dần: [1, 2, 0] c = [10, 10, 3]
bước 0: viên 1, c= 10, Alice cộng a[1]=4   -> A=4, B=0
bước 1: viên 2, c= 10, Bob   cộng b[2]=7   -> A=4, B=7
bước 2: viên 0, c=  3, Alice cộng a[0]=2   -> A=6, B=7
kết quả: -1


-1

In [15]:
trace_sort([1, 3], [2, 1])

thứ tự theo c giảm dần: [1, 0] c = [4, 3]
bước 0: viên 1, c=  4, Alice cộng a[1]=3   -> A=3, B=0
bước 1: viên 0, c=  3, Bob   cộng b[0]=2   -> A=3, B=2
kết quả: 1


1

Trace thêm Cách 1 (minimax) để thấy giá trị `f(mask)` ở từng trạng thái trên ví dụ 1. Mỗi dòng in ra trạng thái, người đi, và nước đi tốt nhất.

In [16]:
def trace_brute(a, b):
    n = len(a)
    memo = {}
    def f(mask, depth):
        if mask == 0:
            return 0
        if mask in memo:
            return memo[mask]
        taken = n - bin(mask).count("1")
        who = "Alice" if taken % 2 == 0 else "Bob"
        vals = a if taken % 2 == 0 else b
        best, arg = None, None
        for i in range(n):
            if mask >> i & 1:
                v = vals[i] - f(mask ^ (1 << i), depth + 1)
                if best is None or v > best:
                    best, arg = v, i
        memo[mask] = best
        print("  " * depth + f"mask={mask:0{n}b} ({who} đi): lấy viên {arg}, f={best}")
        return best
    d = f((1 << n) - 1, 0)
    print("hiệu tối ưu (Alice - Bob) =", d)

trace_brute([1, 3], [2, 1])

  mask=10 (Bob đi): lấy viên 1, f=1
  mask=01 (Bob đi): lấy viên 0, f=2
mask=11 (Alice đi): lấy viên 1, f=1
hiệu tối ưu (Alice - Bob) = 1


## 9. Code nộp LeetCode

Chọn Cách 2 (ngắn, dễ đọc, `O(n log n)` thừa đủ cho `n ≤ 10⁵`).

In [17]:
class Solution:
    def stoneGameVI(self, aliceValues: List[int], bobValues: List[int]) -> int:
        order = sorted(range(len(aliceValues)),
                       key=lambda i: aliceValues[i] + bobValues[i], reverse=True)
        A = sum(aliceValues[i] for i in order[::2])
        B = sum(bobValues[i] for i in order[1::2])
        return (A > B) - (A < B)

sol = Solution()
for (a, b), want in examples:
    got = sol.stoneGameVI(a, b)
    print(got, "kỳ vọng", want, "OK" if got == want else "SAI")

1 kỳ vọng 1 OK
0 kỳ vọng 0 OK
-1 kỳ vọng -1 OK


## 10. Vì sao thuật toán đúng

**Bước 1 — quy về một trò chơi "tổng chung".** Ở mục 6 ta có `A − B = Σ_{S_A} c − Σ b` và `B − A = Σ_{S_B} c − Σ a`, với `c[i] = a[i] + b[i]`. Vì `Σ_{S_A} c + Σ_{S_B} c = Σ c` cố định, Alice muốn **lớn** `Σ_{S_A} c`, Bob muốn **lớn** `Σ_{S_B} c` ⇔ **nhỏ** `Σ_{S_A} c`. Đây là một trò chơi **tổng bằng không** trên các "đồng xu" giá trị `c[i]`: hai người lần lượt nhặt đồng xu, mỗi người muốn tổng của mình lớn nhất.

**Bước 2 — trong trò chơi nhặt xu tự do, lấy đồng lớn nhất là tối ưu (exchange argument).** Giả sử đến lượt một người và đồng lớn nhất còn lại là `M`. Xét một chiến lược tối ưu bất kỳ mà lượt này lấy `x < M`. Đồng `M` rồi sẽ thuộc về ai đó:
- nếu về sau chính người này lấy `M`, đổi chỗ hai lượt (lấy `M` bây giờ, `x` lúc đó) không đổi tập của ai ⇒ không tệ hơn;
- nếu đối thủ lấy `M`, thì đổi sang lấy `M` ngay: đối thủ không bao giờ có `M`, tối đa họ lấy `x` thay thế; tổng của mình tăng ít nhất `M − x ≥ 0`.

Lặp lại lập luận cho từng lượt ⇒ chiến lược "luôn lấy `c` lớn nhất" là tối ưu cho **cả hai** người. Hệ quả: Alice nhận các vị trí chẵn của dãy `c` đã sắp xếp giảm dần.

**Bước 3 — hòa giữa các viên cùng `c`.** Nếu `c[i] = c[j]`, việc ai lấy viên nào trong hai viên không đổi `Σ_{S_A} c` ⇒ không đổi `A − B`. Vì vậy thứ tự bên trong xô (Cách 3) hay cách phá hòa khi sắp xếp (Cách 2) đều cho cùng kết quả.

Mục 12 kiểm chứng bằng cách so với minimax vét cạn trên hàng nghìn input ngẫu nhiên.

## 11. Phân tích Time / Space

Gọi `n` là số viên đá, `V = 200` là tổng lớn nhất có thể của `a[i] + b[i]`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Minimax bitmask | `O(n · 2ⁿ)` | `O(2ⁿ)` | chỉ dùng được với `n ≤ ~16` |
| 2. Sắp xếp theo `a+b` | `O(n log n)` | `O(n)` | Timsort; mọi trường hợp như nhau |
| 3. Đếm theo tổng | `O(n + V)` | `O(n + V)` | tận dụng giá trị ≤ 100 |
| 4. Heap mô phỏng | `O(n log n)` | `O(n)` | heapify `O(n)` + `n` pop |

Với `n = 10⁵`, Cách 2 chạy vài chục mili-giây trong Python; Cách 3 còn nhanh hơn chút do không có `log n`.

In [18]:
random.seed(7)
n = 10**5
a = [random.randint(1, 100) for _ in range(n)]
b = [random.randint(1, 100) for _ in range(n)]
for name, fn in [("sort", stone_game_sort), ("count", stone_game_count), ("heap", stone_game_heap)]:
    t0 = time.perf_counter(); r = fn(a, b)
    print(f"{name:5s}: kết quả {r:2d}, {time.perf_counter() - t0:.3f}s")

sort : kết quả  1, 0.019s
count: kết quả  1, 0.011s


heap : kết quả  1, 0.100s


## 12. Bộ test

Gồm: ví dụ đề, edge case (`n = 1`, mọi giá trị bằng nhau, Alice thích viên khác Bob), phản ví dụ cho các tham lam sai, và đối chiếu ngẫu nhiên với minimax (seed cố định, `n ≤ 9`, giá trị nằm trong `[1, 100]` đúng ràng buộc).

In [19]:
all_fns = [stone_game_brute, stone_game_sort, stone_game_count, stone_game_heap,
           lambda a, b: Solution().stoneGameVI(a, b)]

# ví dụ đề
for (a, b), want in examples:
    for fn in all_fns:
        assert fn(a, b) == want

# edge case
edge = [
    (([5], [100]), 1),          # n = 1: Alice lấy viên duy nhất
    (([1, 1], [1, 1]), 0),      # hai viên như nhau -> hòa
    (([100, 1], [1, 100]), 0),  # mỗi người được viên mình thích (đều 100) -> c bằng nhau -> hòa
    (([3, 3, 3], [3, 3, 3]), 1),  # n lẻ, Alice được thêm một viên
]
for (a, b), want in edge:
    assert stone_game_brute(a, b) == want, (a, b)
    for fn in all_fns:
        assert fn(a, b) == want, (fn, a, b)
print("ví dụ đề + edge case: OK")

ví dụ đề + edge case: OK


In [20]:
# đối chiếu ngẫu nhiên với minimax
random.seed(2024)
cnt = 0
for _ in range(1500):
    n = random.randint(1, 9)
    hi = random.choice([3, 10, 100])
    a = [random.randint(1, hi) for _ in range(n)]
    b = [random.randint(1, hi) for _ in range(n)]
    want = stone_game_brute(a, b)
    for fn in all_fns[1:]:
        assert fn(a, b) == want, (a, b, fn)
    cnt += 1
print(f"{cnt} bộ ngẫu nhiên khớp minimax")

1500 bộ ngẫu nhiên khớp minimax


In [21]:
# kiểm tra mạnh hơn: không chỉ dấu mà cả HIỆU tối ưu phải bằng Σ_{S_A} c - Σ b
def best_diff_brute(a, b):
    n = len(a)
    @lru_cache(maxsize=None)
    def f(mask):
        if mask == 0:
            return 0
        vals = a if (n - bin(mask).count("1")) % 2 == 0 else b
        return max(vals[i] - f(mask ^ (1 << i)) for i in range(n) if mask >> i & 1)
    return f((1 << n) - 1)

def best_diff_greedy(a, b):
    c = sorted((x + y for x, y in zip(a, b)), reverse=True)
    return sum(c[::2]) - sum(b)

random.seed(99)
for _ in range(800):
    n = random.randint(1, 9)
    a = [random.randint(1, 100) for _ in range(n)]
    b = [random.randint(1, 100) for _ in range(n)]
    assert best_diff_brute(a, b) == best_diff_greedy(a, b)
print("hiệu tối ưu khớp công thức Σ c[chẵn] - Σ b: OK")

hiệu tối ưu khớp công thức Σ c[chẵn] - Σ b: OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Mỗi người chỉ lấy viên **mình** thích nhất

Trực giác "tham điểm cho mình" bỏ qua việc tước điểm của đối thủ.

In [22]:
def wrong_selfish(a, b):
    left = set(range(len(a)))
    A = B = 0
    alice = True
    while left:
        if alice:
            i = max(left, key=lambda k: a[k]); A += a[i]
        else:
            i = max(left, key=lambda k: b[k]); B += b[i]
        left.remove(i); alice = not alice
    return (A > B) - (A < B)

a, b = [2, 4, 3], [1, 6, 7]
print("ví dụ 3 vẫn đúng tình cờ:", wrong_selfish(a, b), "| brute:", stone_game_brute(a, b))
a, b = [5, 4], [1, 10]
print("phản ví dụ a=[5,4], b=[1,10]: sai =", wrong_selfish(a, b), "| đúng =", stone_game_brute(a, b))

ví dụ 3 vẫn đúng tình cờ: -1 | brute: -1
phản ví dụ a=[5,4], b=[1,10]: sai = -1 | đúng = 1


Với `a=[5,4], b=[1,10]`: Alice ích kỷ lấy viên 0 (5 điểm), Bob lấy viên 1 (10 điểm) ⇒ Bob thắng. Nhưng nếu Alice lấy viên 1 (4 điểm, tước 10 của Bob), Bob chỉ còn 1 điểm ⇒ Alice thắng. **Sửa:** sắp xếp theo `a[i] + b[i]` (Cách 2).

### Lỗi 2 — Sắp xếp theo `a[i] − b[i]`

Nghe có vẻ hợp lý ("lợi thế tương đối"), nhưng dấu trừ sai: lấy viên `i` làm hiệu tăng `a[i]` **cộng** thêm `b[i]` (vì Bob không lấy được nữa).

In [23]:
def wrong_diff(a, b):
    order = sorted(range(len(a)), key=lambda i: a[i] - b[i], reverse=True)
    A = sum(a[i] for i in order[::2]); B = sum(b[i] for i in order[1::2])
    return (A > B) - (A < B)

random.seed(5)
for _ in range(2000):
    n = random.randint(2, 6)
    a = [random.randint(1, 10) for _ in range(n)]
    b = [random.randint(1, 10) for _ in range(n)]
    if wrong_diff(a, b) != stone_game_brute(a, b):
        print("phản ví dụ:", a, b, "| sai =", wrong_diff(a, b), "| đúng =", stone_game_brute(a, b))
        break

phản ví dụ: [6, 4, 9, 4] [3, 4, 7, 5] | sai = -1 | đúng = 1


**Sửa:** khóa sắp xếp là `a[i] + b[i]`.

### Lỗi 3 — Cộng nhầm mảng: cả Alice và Bob đều cộng `aliceValues`

Khi viết `sum(aliceValues[i] for i in order[1::2])` cho Bob (copy-paste), kết quả so sánh sai vì Bob tính điểm theo **cách định giá của Bob**.

In [24]:
def wrong_same_array(a, b):
    order = sorted(range(len(a)), key=lambda i: a[i] + b[i], reverse=True)
    A = sum(a[i] for i in order[::2])
    B = sum(a[i] for i in order[1::2])      # SAI: phải là b[i]
    return (A > B) - (A < B)

a, b = [2, 4, 3], [1, 6, 7]
print("sai =", wrong_same_array(a, b), "| đúng =", stone_game_sort(a, b))

sai = 1 | đúng = -1


**Sửa:** `B = sum(b[i] for i in order[1::2])`.

### Lỗi 4 — Sắp xếp **riêng** hai mảng rồi zip lại

`sorted(a)` và `sorted(b)` làm mất liên kết "viên `i` có cặp giá trị `(a[i], b[i])`". Phải sắp xếp **chỉ số** (hoặc cặp) theo một khóa chung.

In [25]:
def wrong_sort_separately(a, b):
    sa, sb = sorted(a, reverse=True), sorted(b, reverse=True)
    A = sum(sa[::2]); B = sum(sb[1::2])
    return (A > B) - (A < B)

random.seed(8)
for _ in range(3000):
    n = random.randint(2, 5)
    a = [random.randint(1, 10) for _ in range(n)]
    b = [random.randint(1, 10) for _ in range(n)]
    if wrong_sort_separately(a, b) != stone_game_brute(a, b):
        print("phản ví dụ:", a, b, "| sai =", wrong_sort_separately(a, b), "| đúng =", stone_game_brute(a, b))
        break

phản ví dụ: [1, 7, 5, 7] [5, 7, 3, 7] | sai = 1 | đúng = 0


**Sửa:** `order = sorted(range(n), key=lambda i: a[i] + b[i], reverse=True)` rồi dùng cùng chỉ số `i` cho cả hai mảng.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Dấu hiệu:** trò chơi hai người, **chọn tự do** bất kỳ phần tử nào (không bị ràng buộc đầu/cuối), mỗi người định giá khác nhau ⇒ thử khóa `a[i] + b[i]` ("được + tước").
- Nếu phải chọn **đầu hoặc cuối dãy** (Stone Game I, VII, …) thì tham lam thường sai ⇒ DP khoảng `dp[i][j]`.
- Viết lại mục tiêu `A − B` thành "tổng trên tập của mình − hằng số" là mẹo chung để quy trò chơi không đối xứng về trò chơi tổng-không.
- Luôn có sẵn một **minimax bitmask** 10 dòng để kiểm tra tham lam trên `n ≤ 10`.
- Giá trị bị chặn nhỏ (≤ 100) ⇒ cân nhắc counting sort.

## 15. Biến thể của bài

1. **Stone Game VI, trả về hiệu điểm** thay vì dấu: `Σ c[vị trí chẵn] − Σ b` (đã kiểm ở mục 12).
2. **Bob đi trước**: đổi vai — Bob nhận vị trí chẵn của dãy đã sắp.
3. **Chỉ chơi `k` lượt** (mỗi người lấy tối đa vài viên, phần còn lại bỏ): vẫn lấy `c` lớn nhất trước vì lập luận đổi chỗ vẫn đúng.
4. **Stone Game VII** (LeetCode 1690): lấy đầu/cuối, điểm = tổng phần còn lại ⇒ DP khoảng.
5. **Stone Game I/II/III**: các luật lấy khác nhau, đa số cần DP minimax.
6. **"Hai người chọn đồ chung"** trong bài toán phân chia: cùng ý tưởng `a + b` (ví dụ Codeforces "Alice and Bob game with two value lists").

In [26]:
def best_diff(a, b, alice_first=True):
    """Biến thể: trả về hiệu A - B; cho phép chọn ai đi trước."""
    c = sorted(range(len(a)), key=lambda i: a[i] + b[i], reverse=True)
    first, second = (c[::2], c[1::2]) if alice_first else (c[1::2], c[::2])
    return sum(a[i] for i in first) - sum(b[i] for i in second)

print("ví dụ 3, Alice đi trước: hiệu =", best_diff([2, 4, 3], [1, 6, 7]))
print("ví dụ 3, Bob đi trước:   hiệu =", best_diff([2, 4, 3], [1, 6, 7], alice_first=False))

ví dụ 3, Alice đi trước: hiệu = -1
ví dụ 3, Bob đi trước:   hiệu = -4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Minimax bitmask | thử mọi nước, ghi nhớ theo tập còn lại | `O(n·2ⁿ)` | `O(2ⁿ)` | kiểm chứng, `n ≤ 15` |
| 2. Sort theo `a+b` | được + tước, lấy lớn nhất | `O(n log n)` | `O(n)` | **mặc định khi nộp** |
| 3. Counting theo tổng | xô 2..200 | `O(n + V)` | `O(n + V)` | giá trị bị chặn nhỏ |
| 4. Heap mô phỏng | rút max mỗi lượt | `O(n log n)` | `O(n)` | biến thể có thêm/bớt viên động |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `aliceValues = [9, 1, 1]`, `bobValues = [1, 9, 1]`. Kết quả là gì?

**Bài 2 (tính tay):** với `a = [3, 7, 2, 5]`, `b = [6, 1, 4, 5]`, lập bảng `c = a + b`, thứ tự chọn, điểm A, B và hiệu.

**Bài 3 (điền TODO):** hoàn thiện hàm `alice_score_only(a, b)` trả về **điểm của Alice** khi cả hai chơi tối ưu.

**Bài 4 (tự viết lại):** viết lại Cách 3 (đếm theo tổng) nhưng **không lưu danh sách chỉ số**, chỉ lưu cho mỗi tổng `s` danh sách giá trị `a[i]` (và `b[i]`). Gợi ý: thứ tự trong xô không quan trọng.

**Bài 5 (mini challenge):** trò chơi **ba người** Alice → Bob → Carol, mỗi người muốn tối đa "điểm mình − tổng điểm hai người kia"? Hãy viết minimax bitmask cho `n ≤ 8` và thử xem tham lam theo `a+b+c` có còn đúng không (gợi ý: trò chơi không còn tổng-không giữa hai phe, hãy thực nghiệm).

In [27]:
def alice_score_only(a, b):
    # TODO: sắp xếp chỉ số theo a[i] + b[i] giảm dần, cộng a[i] ở vị trí chẵn
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Tính `c = [10, 10, 2]`. (2) Alice lấy một trong hai viên tổng 10, Bob lấy viên còn lại tổng 10, Alice lấy viên cuối. (3) Alice tốt nhất lấy viên 0 (9) + viên 2 (1) = 10, Bob được 9 ⇒ `1`.

**Bài 2:** (1) `c = [9, 8, 6, 10]`. (2) Thứ tự: viên 3, 0, 1, 2. (3) Alice: viên 3 và 1 ⇒ `5 + 7 = 12`; Bob: viên 0 và 2 ⇒ `6 + 4 = 10`.

**Bài 3:** (1) Giống Cách 2. (2) Chỉ trả về `A`. (3) `sum(a[i] for i in order[::2])`.

**Bài 4:** (1) `bucket[s]` là list các cặp `(a[i], b[i])`. (2) Biến `turn` chạy liên tục qua mọi xô. (3) Duyệt `s` từ 200 xuống 2.

**Bài 5:** (1) Trạng thái `mask`, người đi = (số viên đã lấy) mod 3. (2) Hàm trả về **vector 3 điểm**; người đi chọn nước tối đa (điểm mình − hai người kia). (3) So sánh với tham lam trên nhiều input ngẫu nhiên; nếu tìm được phản ví dụ, in ra.

## 19. Đáp án tham khảo

In [28]:
# Bài 1
assert stone_game_brute([9, 1, 1], [1, 9, 1]) == 1 == stone_game_sort([9, 1, 1], [1, 9, 1])

# Bài 2
a, b = [3, 7, 2, 5], [6, 1, 4, 5]
order = sorted(range(4), key=lambda i: a[i] + b[i], reverse=True)
A = sum(a[i] for i in order[::2]); B = sum(b[i] for i in order[1::2])
assert order == [3, 0, 1, 2] and (A, B) == (12, 10)
assert best_diff_brute(a, b) == 2

# Bài 3
def alice_score_only_ans(a, b):
    order = sorted(range(len(a)), key=lambda i: a[i] + b[i], reverse=True)
    return sum(a[i] for i in order[::2])
assert alice_score_only_ans([2, 4, 3], [1, 6, 7]) in (5, 6)   # phụ thuộc cách phá hòa, hiệu vẫn = -1

# Bài 4
def stone_game_count_values(a, b):
    bucket = [[] for _ in range(201)]
    for x, y in zip(a, b):
        bucket[x + y].append((x, y))
    A = B = turn = 0
    for s in range(200, 1, -1):
        for x, y in bucket[s]:
            if turn == 0: A += x
            else: B += y
            turn ^= 1
    return (A > B) - (A < B)

random.seed(11)
for _ in range(500):
    n = random.randint(1, 8)
    a = [random.randint(1, 100) for _ in range(n)]
    b = [random.randint(1, 100) for _ in range(n)]
    assert stone_game_count_values(a, b) == stone_game_brute(a, b)
print("Bài 1-4: OK")

Bài 1-4: OK


In [29]:
# Bài 5 — ba người, thực nghiệm
def three_player_brute(vals):
    """vals = [a, b, c]; trả về vector điểm khi mỗi người tối đa (mình - hai người kia)."""
    n = len(vals[0])
    @lru_cache(maxsize=None)
    def f(mask):
        if mask == 0:
            return (0, 0, 0)
        p = (n - bin(mask).count("1")) % 3
        best, best_key = None, None
        for i in range(n):
            if mask >> i & 1:
                sub = list(f(mask ^ (1 << i)))
                sub[p] += vals[p][i]
                key = 2 * sub[p] - sum(sub)      # mình - (tổng hai người kia)
                if best_key is None or key > best_key:
                    best, best_key = tuple(sub), key
        return best
    return f((1 << n) - 1)

def three_player_greedy(vals):
    n = len(vals[0])
    order = sorted(range(n), key=lambda i: vals[0][i] + vals[1][i] + vals[2][i], reverse=True)
    sc = [0, 0, 0]
    for t, i in enumerate(order):
        sc[t % 3] += vals[t % 3][i]
    return tuple(sc)

random.seed(3)
found = None
for _ in range(3000):
    n = random.randint(2, 6)
    vals = [[random.randint(1, 6) for _ in range(n)] for _ in range(3)]
    key = lambda s: [2 * s[p] - sum(s) for p in range(3)]
    if key(three_player_brute(vals))[0] != key(three_player_greedy(vals))[0]:
        found = vals; break
print("phản ví dụ cho tham lam 3 người:", found)

phản ví dụ cho tham lam 3 người: [[4, 6, 5, 4, 2], [3, 1, 1, 2, 4], [2, 3, 6, 4, 6]]


Kết luận Bài 5: với ba người, mục tiêu của các người chơi **không còn là đối nghịch tuyệt đối** giữa hai phe, nên lập luận đổi chỗ ở mục 10 không áp dụng; thực nghiệm (nếu in ra phản ví dụ) cho thấy tham lam `a+b+c` không còn đảm bảo. Đây là lý do bài gốc chỉ có hai người.

## 20. Template nhớ nhanh + Checklist

```python
# Trò chơi "lấy tự do", định giá khác nhau  ->  tham lam theo được + tước
order = sorted(range(n), key=lambda i: a[i] + b[i], reverse=True)
A = sum(a[i] for i in order[::2])      # người đi trước
B = sum(b[i] for i in order[1::2])     # người đi sau
return (A > B) - (A < B)
```

**Checklist trước khi submit:**
- [ ] Khóa sắp xếp là `a[i] + b[i]` (không phải `a[i]`, không phải `a[i] - b[i]`).
- [ ] Sắp xếp **chỉ số**, không sắp xếp riêng hai mảng.
- [ ] Alice cộng `aliceValues`, Bob cộng `bobValues`.
- [ ] Alice nhận vị trí **chẵn** `[::2]` (đi trước), Bob nhận `[1::2]`.
- [ ] Trả về đúng `1 / 0 / -1`, không trả về hiệu.
- [ ] Đã thử `n = 1` và trường hợp hai viên cùng tổng.